# CosmoCalc: Python Calculator for Cosmogenic Nuclide Exposure Ages

**Authors:** Michal Ben-Israel  
**Based on:** Balco et al. (2008) CRONUS-Earth calculator v3  
**License:** GNU GPL v2  

## Overview
Python implementation of 10Be (and 26Al) surface exposure age and denudation rate 
calculator. Translated and updated from the MATLAB-based CRONUS-Earth framework 
(Balco et al., 2008) with the following updates:
- LSDn scaling scheme (Lifton et al., 2014)
- Updated production rates (Borchers et al., 2016)
- Updated muon production (Balco, 2017)

## References
- Balco et al. (2008) Quaternary Geochronology 3, 174-195
- Lifton et al. (2014) EPSL 386, 149-160
- Borchers et al. (2016) Quaternary Geochronology 31, 188-198
- Balco (2017) Quaternary Geochronology 39, 150-173

In [28]:
import numpy as np
import scipy.io as sio
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator, interp1d
from scipy.integrate import cumulative_trapezoid

In [29]:
# ================================================
# CONSTANTS
# Borchers et al. (2016) Table 7
# SLHL spallation production rates (atoms/g quartz/yr)
# ================================================

# 10Be production rates
P10_St   = 4.01   # Stone (2000)
P10_Lm   = 4.00   # Lal/Stone paleomagnetic
P10_LSDn = 3.92   # Lifton-Sato-Dunai (preferred, v3 default)

# 26Al production rates
P26_St   = 27.93
P26_Lm   = 27.93
P26_LSDn = 28.54

# Decay constants (yr-1)
l10 = 4.9975e-7   # 10Be, half-life 1.387 Ma (Korschinek et al. 2010)
l26 = 9.83e-7     # 26Al, half-life 0.705 Ma (Nishiizumi 2004)

# Attenuation lengths (g/cm2)
Lsp  = 160.0      # spallation (Gosse & Phillips 2001)
Lmu  = 1500.0     # muons

## 1. Load data files

In [ ]:
# Load data files
ncep      = sio.loadmat('../data/NCEP2.mat',            squeeze_me=True, struct_as_record=False)
pmag07    = sio.loadmat('../data/PMag_Mar07.mat',        squeeze_me=True, struct_as_record=False)
pmag12    = sio.loadmat('../data/Pmag_Sep12.mat',        squeeze_me=True, struct_as_record=False)
lsd       = sio.loadmat('../data/consts_LSD.mat',        squeeze_me=True, struct_as_record=False)
consts_v23 = sio.loadmat('../data/al_be_consts_v23.mat', squeeze_me=True, struct_as_record=False)


## 2. Atmosphere
Converts sample elevation to atmospheric pressure using the NCEP reanalysis 
gridded sea-level pressure and temperature data (NCEPatm_2.m equivalent).

In [19]:
def ncep_pressure(lat, lon, elv):
    """
    Atmospheric pressure from elevation using NCEP reanalysis.
    Equivalent to NCEPatm_2.m (Balco v2.3).

    Inputs:
        lat : latitude (degrees N)
        lon : longitude (degrees E, 0-360)
        elv : elevation (m)
    Returns:
        pressure : atmospheric pressure (hPa)
    """
    NCEPlat   = ncep['NCEPlat']
    NCEPlon   = ncep['NCEPlon']
    meanslp   = ncep['meanslp']
    meant1000 = ncep['meant1000']

    interp_slp = RegularGridInterpolator((NCEPlat, NCEPlon), meanslp)
    interp_T   = RegularGridInterpolator((NCEPlat, NCEPlon), meant1000)

    slp = interp_slp([[lat, lon]])[0]
    T   = interp_T([[lat, lon]])[0] + 273.15

    gmr  = -0.03417
    dtdz =  0.0065

    pressure = slp * np.exp((gmr / dtdz) * (np.log(T) - np.log(T - elv * dtdz)))
    return pressure

# Test
print(f"Pressure at 37N, 240E, 3000m: {ncep_pressure(37.0, 240.0, 3000.0):.2f} hPa")

## Cell 6 — Snow data (SNOTEL)
Snow shielding calculations based on Bason on Gosse & Phillips (2001)
Source: NRCS SNOTEL North Lake station (NTH), CA
Elevation: 9280 ft | Period of record: 1930-2026
Monthly SWE (snow water equivalent, cm)

In [31]:
df_snow = pd.read_csv('../data/NTH_snow.csv', skiprows=62, header=[0,1])

swe_cols = [('Jan', 'Snow Water Equivalent (in) Start of Month Values'),
            ('Feb', 'Snow Water Equivalent (in) Start of Month Values'),
            ('Mar', 'Snow Water Equivalent (in) Start of Month Values'),
            ('Apr', 'Snow Water Equivalent (in) Start of Month Values'),
            ('May', 'Snow Water Equivalent (in) Start of Month Values'),
            ('Jun', 'Snow Water Equivalent (in) Start of Month Values')]

swe_cm = np.zeros(12)
for i, col in enumerate(swe_cols):
    swe = pd.to_numeric(df_snow[col], errors='coerce')
    valid = swe.notna() & (swe > 0)
    if valid.sum() > 0:
        swe_cm[i] = swe[valid].mean() * 2.54

## 3. Cover Corrections
Calculates the scaling factor for sample thickness and surface cover 
(snow, soil, till). Equivalent to the thickness correction in Balco v2.3,
extended to include multiple cover layers.

In [22]:
def snow_shielding(depths_monthly, densities_monthly, L=160.0):
    """
    Monthly snow shielding correction (after Gosse & Phillips 2001).
    Inputs:
        depths_monthly    : array of 12 monthly snow depths (cm)
        densities_monthly : array of 12 monthly snow densities (g/cm3)
        L                 : spallation attenuation length (g/cm2)
    Returns:
        SF_snow : dimensionless snow shielding factor
    """
    depths_monthly    = np.array(depths_monthly)
    densities_monthly = np.array(densities_monthly)
    return np.mean(np.exp(-depths_monthly * densities_monthly / L))


def cover_correction(rho_rock, thickness,
                     depth_soil=0.0, rho_soil=1.6,
                     depth_till=0.0, rho_till=1.9,
                     depths_snow_monthly=None,
                     densities_snow_monthly=None,
                     L=160.0):
    """
    Thickness and cover scaling factor for n samples.
    Vectorized — accepts scalars or arrays of length n.

    Inputs:
        rho_rock               : rock density (g/cm3), scalar or array
        thickness              : sample thickness (cm), scalar or array
        depth_soil             : soil depth (cm), scalar or array
        rho_soil               : soil density (g/cm3), scalar or default
        depth_till             : till depth (cm), scalar or array
        rho_till               : till density (g/cm3), scalar or default
        depths_snow_monthly    : (n,12) array of monthly snow depths (cm)
                                  or (12,) if same for all samples
        densities_snow_monthly : (n,12) array of monthly snow densities
                                  or (12,) if same for all samples
        L                      : attenuation length (g/cm2), default 160

    Returns:
        SF_thickness : thickness+soil+till scaling factor, array of length n
        SF_snow      : snow shielding factor, array of length n (1.0 if no snow)
    """
    rho_rock   = np.atleast_1d(np.array(rho_rock,   dtype=float))
    thickness  = np.atleast_1d(np.array(thickness,  dtype=float))
    depth_soil = np.atleast_1d(np.array(depth_soil, dtype=float))
    depth_till = np.atleast_1d(np.array(depth_till, dtype=float))
    n = len(rho_rock)

    # Solid cover (rock thickness + soil + till)
    total_solid = (rho_rock  * thickness +
                   rho_soil  * depth_soil +
                   rho_till  * depth_till)

    SF_thickness = (L / total_solid) * (1 - np.exp(-total_solid / L))

    # Snow shielding (monthly, per sample)
    if depths_snow_monthly is None:
        SF_snow = np.ones(n)
    else:
        depths_snow    = np.atleast_2d(np.array(depths_snow_monthly,    dtype=float))
        densities_snow = np.atleast_2d(np.array(densities_snow_monthly, dtype=float))
        # broadcast (12,) to (n,12) if needed
        if depths_snow.shape[0] == 1:
            depths_snow    = np.repeat(depths_snow,    n, axis=0)
            densities_snow = np.repeat(densities_snow, n, axis=0)
        SF_snow = np.mean(np.exp(-depths_snow * densities_snow / L), axis=1)

    return SF_thickness, SF_snow


# --- Tests ---
# Single sample, no cover
SF_t, SF_s = cover_correction(rho_rock=2.7, thickness=2.0)
print(f"Single sample, no cover:  SF_thickness={SF_t[0]:.4f}, SF_snow={SF_s[0]:.4f}")

# Single sample, 20cm soil
SF_t, SF_s = cover_correction(rho_rock=2.7, thickness=2.0, depth_soil=20.0)
print(f"Single sample, 20cm soil: SF_thickness={SF_t[0]:.4f}, SF_snow={SF_s[0]:.4f}")

# Multiple samples, shared monthly snow
depths_mo    = [75,75,75,75,75,75, 0, 0, 0, 0, 0, 0]
densities_mo = [0.25]*6 + [0.0]*6
SF_t, SF_s = cover_correction(
    rho_rock  = [2.7, 2.7, 2.65],
    thickness = [2.0, 3.0, 1.5],
    depth_soil= [0.0, 10.0, 5.0],
    depths_snow_monthly    = depths_mo,
    densities_snow_monthly = densities_mo
)
for i in range(3):
    print(f"Sample {i+1}: SF_thickness={SF_t[i]:.4f}, SF_snow={SF_s[i]:.4f}")

Single sample, no cover:  SF_thickness=0.9833, SF_snow=1.0000
Single sample, 20cm soil: SF_thickness=0.8917, SF_snow=1.0000
Sample 1: SF_thickness=0.9833, SF_snow=0.9447
Sample 2: SF_thickness=0.9283, SF_snow=0.9447
Sample 3: SF_thickness=0.9635, SF_snow=0.9447


## 4. Load Sample Data
Reads sample data from user-provided Excel file.
Snow correction uses shared monthly depths (one set per study area).

In [21]:
def load_samples(filepath, sheet_samples='samples', sheet_snow='snow'):
    """
    Load sample data and snow correction from Excel file.
    
    Expected columns in 'samples' sheet:
        sample, lat, lon, elv, N10, delN10, 
        shieldcorr, rho, thickness, depth_soil, depth_till, site
    
    Expected columns in 'snow' sheet:
        month (1-12), depth_cm, density_gcm3
    
    Returns:
        df       : DataFrame of sample data
        snow_depths    : array (12,) of monthly snow depths (cm)
        snow_densities : array (12,) of monthly snow densities (g/cm3)
    """
    # Load samples
    df = pd.read_excel(filepath, sheet_name=sheet_samples)
    
    # Load snow if sheet exists
    try:
        snow = pd.read_excel(filepath, sheet_name=sheet_snow)
        snow_depths    = snow['depth_cm'].values.astype(float)
        snow_densities = snow['density_gcm3'].values.astype(float)
    except:
        print("No snow sheet found - assuming no snow cover.")
        snow_depths    = np.zeros(12)
        snow_densities = np.zeros(12)
    
    print(f"Loaded {len(df)} samples.")
    print(f"Sites: {df['site'].unique()}")
    print(f"Columns: {list(df.columns)}")
    
    return df, snow_depths, snow_densities

In [24]:
# Parse SNOTEL data
# Skip 62 comment lines, then two header rows
df_snow = pd.read_csv('../data/NTH_snow.csv', 
                      skiprows=62, 
                      header=[0,1])

# Column indices for each month:
# Jan: depth=col2, SWE=col3
# Feb: depth=col5, SWE=col6
# Mar: depth=col8, SWE=col9
# Apr: depth=col11, SWE=col12
# May: depth=col14, SWE=col15
# Jun: depth=col17, SWE=col18

depth_cols = [2, 5, 8, 11, 14, 17]
swe_cols   = [3, 6, 9, 12, 15, 18]
months     = ['Jan','Feb','Mar','Apr','May','Jun']

depths_cm  = np.zeros(12)
densities  = np.zeros(12)

for i, (dc, sc) in enumerate(zip(depth_cols, swe_cols)):
    depth = pd.to_numeric(df_snow.iloc[:, dc], errors='coerce')
    swe   = pd.to_numeric(df_snow.iloc[:, sc], errors='coerce')
    valid = (depth > 0) & depth.notna() & swe.notna()
    if valid.sum() > 0:
        depths_cm[i] = depth[valid].mean() * 2.54
        densities[i] = (swe[valid] / depth[valid]).mean()

# Print results
for i, m in enumerate(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']):
    print(f"{m}: depth={depths_cm[i]:.1f} cm, density={densities[i]:.3f} g/cm3")

Jan: depth=0.0 cm, density=0.000 g/cm3
Feb: depth=0.0 cm, density=0.000 g/cm3
Mar: depth=0.0 cm, density=0.000 g/cm3
Apr: depth=0.0 cm, density=0.000 g/cm3
May: depth=0.0 cm, density=0.000 g/cm3
Jun: depth=0.0 cm, density=0.000 g/cm3
Jul: depth=0.0 cm, density=0.000 g/cm3
Aug: depth=0.0 cm, density=0.000 g/cm3
Sep: depth=0.0 cm, density=0.000 g/cm3
Oct: depth=0.0 cm, density=0.000 g/cm3
Nov: depth=0.0 cm, density=0.000 g/cm3
Dec: depth=0.0 cm, density=0.000 g/cm3


In [26]:
# SWE-based snow shielding (no depth needed)
# s = SWE_cm / L (SWE already in g/cm2 = cm * density)
# Since SWE is water equivalent, rho_water = 1.0, so SWE_cm = SWE_inches * 2.54

months     = ['Jan','Feb','Mar','Apr','May','Jun']
swe_cols   = [('Jan','Snow Water Equivalent (in) Start of Month Values'),
              ('Feb','Snow Water Equivalent (in) Start of Month Values'),
              ('Mar','Snow Water Equivalent (in) Start of Month Values'),
              ('Apr','Snow Water Equivalent (in) Start of Month Values'),
              ('May','Snow Water Equivalent (in) Start of Month Values'),
              ('Jun','Snow Water Equivalent (in) Start of Month Values')]

swe_cm = np.zeros(12)  # Jul-Dec stay zero

for i, col in enumerate(swe_cols):
    swe = pd.to_numeric(df_snow[col], errors='coerce')
    valid = swe.notna() & (swe > 0)
    if valid.sum() > 0:
        swe_cm[i] = swe[valid].mean() * 2.54

print("Mean monthly SWE (cm water equivalent):")
for i, m in enumerate(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']):
    print(f"  {m}: {swe_cm[i]:.2f} cm")

Mean monthly SWE (cm water equivalent):
  Jan: 0.00 cm
  Feb: 18.21 cm
  Mar: 25.98 cm
  Apr: 27.42 cm
  May: 30.59 cm
  Jun: 0.00 cm
  Jul: 0.00 cm
  Aug: 0.00 cm
  Sep: 0.00 cm
  Oct: 0.00 cm
  Nov: 0.00 cm
  Dec: 0.00 cm


In [27]:
def snow_shielding_swe(swe_cm_monthly, L=160.0):
    """
    Snow shielding correction using Snow Water Equivalent.
    SWE already accounts for density, so shielding mass = SWE (g/cm2).
    After Gosse & Phillips (2001), Schildgen et al. (2005).
    
    Inputs:
        swe_cm_monthly : array of 12 monthly SWE values (cm water equivalent)
        L              : spallation attenuation length (g/cm2), default 160
    Returns:
        SF_snow : dimensionless snow shielding factor
    """
    swe = np.array(swe_cm_monthly)
    return np.mean(np.exp(-swe / L))

SF_snow = snow_shielding_swe(swe_cm)
print(f"North Lake snow shielding factor: {SF_snow:.4f}")
print(f"Production rate reduction: {(1-SF_snow)*100:.1f}%")
print(f"\nNote: Based on North Lake SNOTEL station (9280 ft),")
print(f"period of record 1930-2026 (measurements mostly Feb-May).")
print(f"January assumed zero due to sparse historical measurements.")
print(f"June-December assumed zero (no persistent snowpack).")

North Lake snow shielding factor: 0.9509
Production rate reduction: 4.9%

Note: Based on North Lake SNOTEL station (9280 ft),
period of record 1930-2026 (measurements mostly Feb-May).
January assumed zero due to sparse historical measurements.
June-December assumed zero (no persistent snowpack).
